In [1]:
import pandas as pd
import geopandas as gpd
import math
import os

In [2]:
import folium
from folium import Choropleth, Circle, Marker
from folium.plugins import HeatMap, MarkerCluster

In [3]:
import numpy as np
os.environ['SHAPE_RESTORE_SHX'] = 'YES'

In [57]:
inssurance_list = pd.read_csv("C:/Users/Century-PC/Desktop/GAM_assurance/CATNAT_2023_2025.xlsx - 2023.csv")

In [58]:
inssurance_list.head()

,NUMERO_POLICE,CODE_SOUS_BRANCHE,NUM_AVNT_COURS,DATE_EFFET,DATE_EXPIRATION,TYPE,WILAYA,COMMUNE,CAPITAL_ASSURE,PRIME_NETTE
0,16330013230012,3302,0,03/01/2023,02/01/2024,1 - Installation Industrielle,9 - BLIDA,111 - BIRTOUTA,1000000,"2500,000"
1,34303003230006,3302,0,04/01/2023,03/01/2024,1 - Installation Industrielle,34 - B.B ARRERIDJ,368 - BORDJ BOU ARRERIDJ,1000000,"2500,000"
2,19306003230396,3302,0,10/05/2023,09/05/2024,1 - Installation Industrielle,19 - SETIF,1158 - EL EULMA,60000000,"25800,000"
3,23304003230170,3302,0,04/12/2023,03/12/2024,1 - Installation Industrielle,23 - ANNABA,155 - AIN BERDA,50000000,"-21500,000"
4,23304003230170,3302,0,04/12/2023,03/12/2024,1 - Installation Industrielle,23 - ANNABA,155 - AIN BERDA,50000000,"21500,000"


In [59]:
import json

In [60]:
with open("Commune_Of_Algeria.json", "r", encoding="utf-8") as f:
    communes_raw = json.load(f)
 
df_communes = pd.DataFrame(communes_raw)
df_communes["wilaya_id"] = pd.to_numeric(df_communes["wilaya_id"], errors="coerce")

In [61]:
df_communes = df_communes.drop(columns='ar_name')

In [62]:
df_communes.head()

,id,post_code,name,wilaya_id,longitude,latitude
0,1,01001,Adrar,1,27.9763317,-0.4841573
1,2,01002,Tamest,1,27.4257197,-0.2807673
2,3,01003,Charouine,1,29.0189483,-0.2690792
3,4,01004,Reggane,1,25.4775826,-4.3665328
4,5,01005,In Zghmir,1,25.8711501,-6.252223


In [63]:
inssurance_list['COMMUNE'] = inssurance_list['COMMUNE'].str.split(' - ').str[1]
inssurance_list['WILAYA'] = inssurance_list['WILAYA'].str.split(' - ').str[1]

In [64]:
df_communes['name'] = df_communes['name'].str.upper()

In [65]:
df_communes.head()

,id,post_code,name,wilaya_id,longitude,latitude
0,1,01001,ADRAR,1,27.9763317,-0.4841573
1,2,01002,TAMEST,1,27.4257197,-0.2807673
2,3,01003,CHAROUINE,1,29.0189483,-0.2690792
3,4,01004,REGGANE,1,25.4775826,-4.3665328
4,5,01005,IN ZGHMIR,1,25.8711501,-6.252223


In [66]:
inssurance_list.head()

,NUMERO_POLICE,CODE_SOUS_BRANCHE,NUM_AVNT_COURS,DATE_EFFET,DATE_EXPIRATION,TYPE,WILAYA,COMMUNE,CAPITAL_ASSURE,PRIME_NETTE
0,16330013230012,3302,0,03/01/2023,02/01/2024,1 - Installation Industrielle,BLIDA,BIRTOUTA,1000000,"2500,000"
1,34303003230006,3302,0,04/01/2023,03/01/2024,1 - Installation Industrielle,B.B ARRERIDJ,BORDJ BOU ARRERIDJ,1000000,"2500,000"
2,19306003230396,3302,0,10/05/2023,09/05/2024,1 - Installation Industrielle,SETIF,EL EULMA,60000000,"25800,000"
3,23304003230170,3302,0,04/12/2023,03/12/2024,1 - Installation Industrielle,ANNABA,AIN BERDA,50000000,"-21500,000"
4,23304003230170,3302,0,04/12/2023,03/12/2024,1 - Installation Industrielle,ANNABA,AIN BERDA,50000000,"21500,000"


In [67]:
df_communes = df_communes.rename(columns={"name": "COMMUNE"}) 

In [68]:
inssurance_merged = pd.merge(inssurance_list, df_communes, on="COMMUNE")

In [69]:
inssurance_merged = inssurance_merged.drop(columns="id")

In [70]:
inssurance_merged

,NUMERO_POLICE,CODE_SOUS_BRANCHE,NUM_AVNT_COURS,DATE_EFFET,DATE_EXPIRATION,TYPE,WILAYA,COMMUNE,CAPITAL_ASSURE,PRIME_NETTE,post_code,wilaya_id,longitude,latitude
0,16330013230012,3302,0,03/01/2023,02/01/2024,1 - Installation Industrielle,BLIDA,BIRTOUTA,1000000,"2500,000",16034,16,36.6424589,2.9745906
1,34303003230006,3302,0,04/01/2023,03/01/2024,1 - Installation Industrielle,B.B ARRERIDJ,BORDJ BOU ARRERIDJ,1000000,"2500,000",34001,34,36.0025,4.7653
2,19306003230396,3302,0,10/05/2023,09/05/2024,1 - Installation Industrielle,SETIF,EL EULMA,60000000,"25800,000",19020,19,36.1657394,5.620148
3,23304003230170,3302,0,04/12/2023,03/12/2024,1 - Installation Industrielle,ANNABA,AIN BERDA,50000000,"-21500,000",23009,23,36.6718947,7.5182507
4,23304003230170,3302,0,04/12/2023,03/12/2024,1 - Installation Industrielle,ANNABA,AIN BERDA,50000000,"21500,000",23009,23,36.6718947,7.5182507
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33094,163230093230046,93301,0,23/03/2023,22/03/2024,Bien immobilier,RELIZANE,RELIZANE,3500000,"2275,000",48001,48,35.6696,0.5493
33095,153040093230051,93301,1,10/08/2023,09/08/2024,Bien immobilier,TIZI OUZOU,TIZI OUZOU,2975000,"0,000",15001,15,36.7001863,4.0059166
33096,163390093230088,93301,0,01/01/2024,31/12/2024,Bien immobilier,ALGER,KOUBA,1368000,"1500,000",16018,16,36.7241467,3.0598317
33097,313120093230026,93301,0,11/04/2023,10/04/2024,Bien immobilier,ORAN,BEN FREHA,2800000,"1820,000",31020,31,35.6283,-0.4202


In [71]:
danger_zones = gpd.read_file('C:/Users/Century-PC/Desktop/GAM_assurance/rpa99_zones_sismiques.shp')

In [72]:
algerian_communes = pd.read_csv('C:/Users/Century-PC/Desktop/GAM_assurance/rpa99_zones_sismiques.csv')

In [74]:
danger_zones['WILAYA'] = algerian_communes['Wilaya']
danger_zones['Level'] = algerian_communes['Danger_level']

In [81]:
danger_zones["WILAYA"] = danger_zones["WILAYA"].str.upper()
danger_zones.head()

,geometry,WILAYA,Level
0,POINT (1.3667 27.8667),ADRAR,Zone 0
1,POINT (1.3317 36.165),CHLEF,Zone III
2,POINT (2.8833 33.8),CHLEF,Zone IIb
3,POINT (7.1128 35.8781),CHLEF,Zone IIb
4,POINT (6.1742 35.555),CHLEF,Zone IIb


In [82]:
inssurance_final = pd.merge(inssurance_list, danger_zones, on="WILAYA")

In [113]:

inssurance_final.head(20)

,NUMERO_POLICE,CODE_SOUS_BRANCHE,NUM_AVNT_COURS,DATE_EFFET,DATE_EXPIRATION,TYPE,WILAYA,COMMUNE,CAPITAL_ASSURE,PRIME_NETTE,geometry,Level
0,16330013230012,3302,0,03/01/2023,02/01/2024,1 - Installation Industrielle,BLIDA,BIRTOUTA,1000000,"2500,000",POINT (4.05 36.7167),Zone III
1,16330013230012,3302,0,03/01/2023,02/01/2024,1 - Installation Industrielle,BLIDA,BIRTOUTA,1000000,"2500,000",POINT (3.0588 36.7525),Zone IIb
2,16330013230012,3302,0,03/01/2023,02/01/2024,1 - Installation Industrielle,BLIDA,BIRTOUTA,1000000,"2500,000",POINT (3.2636 34.67),Zone IIb
3,16330013230012,3302,0,03/01/2023,02/01/2024,1 - Installation Industrielle,BLIDA,BIRTOUTA,1000000,"2500,000",POINT (5.7667 36.82),Zone IIb
4,16330013230012,3302,0,03/01/2023,02/01/2024,1 - Installation Industrielle,BLIDA,BIRTOUTA,1000000,"2500,000",POINT (5.4119 36.19),Zone IIb
5,16330013230012,3302,0,03/01/2023,02/01/2024,1 - Installation Industrielle,BLIDA,BIRTOUTA,1000000,"2500,000",POINT (0.15 34.83),Zone IIb
6,16330013230012,3302,0,03/01/2023,02/01/2024,1 - Installation Industrielle,BLIDA,BIRTOUTA,1000000,"2500,000",POINT (6.9 36.87),Zone IIb
7,16330013230012,3302,0,03/01/2023,02/01/2024,1 - Installation Industrielle,BLIDA,BIRTOUTA,1000000,"2500,000",POINT (-0.6333 35.19),Zone IIb
8,16330013230012,3302,0,03/01/2023,02/01/2024,1 - Installation Industrielle,BLIDA,BIRTOUTA,1000000,"2500,000",POINT (7.7667 36.9),Zone IIb
9,19306003230396,3302,0,10/05/2023,09/05/2024,1 - Installation Industrielle,SETIF,EL EULMA,60000000,"25800,000",POINT (4.76 36.07),Zone IIa


In [ ]:
# Group by Commune
commune_stats = inssurance_final.groupby('COMMUNE').agg({
    'NUMERO_POLICE': lambda x: list(x),    
    'Level': lambda x: x.mode()[0], 
    'geometry': 'first'            
}).rename(columns={'NUMERO_POLICE': 'POLICE_LIST'})

commune_stats['count'] = commune_stats['POLICE_LIST'].apply(len)

# Extract coordinates for Folium
commune_stats['lat'] = commune_stats['geometry'].apply(lambda p: p.y)
commune_stats['lon'] = commune_stats['geometry'].apply(lambda p: p.x)

In [90]:
commune_stats

,POLICE_LIST,Level,geometry,count,lat,lon
COMMUNE,,,,,,
ADEKAR,"[6306003230036, 6306003230094, 6306003230160, ...",Zone IIa,POINT (8.1167 35.4),12,35.400,8.1167
ADRAR,"[23302013240004, 11301003230193, 16330003230381]",Zone 0,POINT (1.8117 35.605),3,35.605,1.8117
AFLOU,"[3202003230142, 3202003230163, 3202003230180, ...",Zone I,POINT (2.8289 36.47),4,36.470,2.8289
AGHBALOU,"[6309003230042, 6309003230034, 6309003230075, ...",Zone IIa,POINT (7.4333 36.46),17,36.460,7.4333
AHL EL KSAR,"[10302003230098, 10302003230220, 1030200323009...",Zone IIa,POINT (7.4333 36.46),7,36.460,7.4333
...,...,...,...,...,...,...
ZERDEZAS,"[25301013230184, 25301013230182]",Zone IIa,POINT (8.3133 36.767),2,36.767,8.3133
ZERIBET EL OUED,"[7201003230055, 25301013230008]",Zone I,POINT (-1.3167 34.88),2,34.880,-1.3167
ZIAMA MANSOURIA JIJEL,"[6308003230002, 6308003230046, 6308003230050, ...",Zone IIa,POINT (8.4833 26.5),7,26.500,8.4833


In [102]:
m_3 = folium.Map(location=[28.0339, 1.6596], tiles='cartodbpositron', zoom_start=6)

In [103]:
color_map = {
    'Zone 0': 'green',
    'Zone I': 'yellow',
    'Zone II': 'orange',
    'Zone IIa': 'orange',
    'Zone IIb': 'orange',
    'Zone III': 'red'
}

In [106]:
mc = MarkerCluster().add_to(m_3)

In [107]:
for commune, row in commune_stats.iterrows():
    # Basic data extraction
    lat, lon = row['lat'], row['lon']
    risk = row['Level']
    count = row['count']
    
    # Skip if coordinates are invalid
    if pd.isna(lat) or pd.isna(lon):
        continue
        
    # Get color
    m_color = color_map.get(risk, 'blue')
    
    # Add Marker with extremely simple popup (to avoid HTML parsing errors)
    folium.Marker(
        location=[lat, lon],
        popup=f"{commune}: {count} buildings",
        icon=folium.Icon(color=m_color, icon='info-sign')
    ).add_to(mc)

# 4. Display
m_3

C:\Users\Century-PC\AppData\Local\Temp\ipykernel_19020\2846979712.py:18: UserWarning: color argument of Icon should be one of: {'orange', 'gray', 'cadetblue', 'green', 'lightgreen', 'blue', 'lightgray', 'darkred', 'pink', 'purple', 'red', 'darkpurple', 'darkgreen', 'white', 'lightred', 'darkblue', 'black', 'lightblue', 'beige'}.
  icon=folium.Icon(color=m_color, icon='info-sign')


In [114]:
wilaya_communes_levels = inssurance_final[["WILAYA", "COMMUNE", "Level"]]

In [117]:
wilaya_communes_levels.drop_duplicates(keep=False)

,WILAYA,COMMUNE,Level
1013,ALGER,SKIKDA,Zone III
1363,ALGER,CONSTANTINE,Zone III
1715,BEJAIA,SOUK OUFELLA,Zone IIa
1955,BOUIRA,HAMMEDI,Zone IIa
1972,ALGER,CHERAGA MILA,Zone III
...,...,...,...
41531,CHLEF,SENDJAS,Zone III
41537,CHLEF,SENDJAS,Zone IIa
41734,BLIDA,SIDI MOUSSA,Zone III
41804,BLIDA,HUSSEIN DEY,Zone III


In [119]:
wilaya_communes_levels.to_csv('commune_levels.csv', index=False)